In [2]:
from workspace.sources.local_datasets.preprocessing.tokenization import NLTKTokenizer
from workspace.sources.local_datasets.preprocessing.cleaning import NoiseReduction, Lemmatization, Stemming
from workspace.sources.experiments.metrics import Loss
from workspace.sources.experiments.experiment import Experiment
from workspace.sources.utils import class_name_to_str
from workspace.sources.local_datasets.dataset import Dataset
from workspace.sources.local_datasets.preprocessing.pipelines import PreprocessingPipeline
from workspace.sources.local_datasets.preprocessing.utils import truncate
from sklearn.model_selection import ParameterGrid
from IPython.display import display, Markdown

from workspace.sources.local_datasets.preprocessing.encoders.encoders import DistilliBERT_Encoder as Encoder
from workspace.sources.models.transformers.bert_based_models import DistilliBERT as Model

import mlflow

mlflow.set_tracking_uri('../../mlruns')

In [3]:
experiment_name = f'prefinal-{class_name_to_str(Model.__name__)}-v2'
print('Experiment:', experiment_name)

Experiment: prefinal-distilli_bert-v2


In [4]:
def conduct_model_experiments(dataset_name,
                              dataset_path,
                              dataset_hparams_grid,
                              model_hparams_grid):
    total_runs = len(dataset_hparams_grid) * len(model_hparams_grid)
    for i, dataset_params in enumerate(dataset_hparams_grid):
        for j, model_hparams in enumerate(model_hparams_grid):
            run_number = i * len(model_hparams_grid) + j + 1
            display(Markdown(f'### Run {run_number} of {total_runs}'))
            dataset = Dataset(dataset_name, dataset_path, **dataset_params)

            model = Model(train_best_model_metric=Loss,
                           training_arguments=model_hparams)

            experiment = Experiment(
                name=experiment_name,
                dataset=dataset,
                model=model)
            experiment.run()
            experiment.prune_artifacts()


In [5]:

# max_tokens_params = [128, 512]
max_tokens_params = [128, 256]

pipelines = []

for max_tokens in max_tokens_params:
    pipelines.extend([PreprocessingPipeline(name='minimal',
                                            iterable=[Encoder(truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction',
                                            iterable=[NoiseReduction(),
                                                      Encoder(truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                            iterable=[NoiseReduction(), NLTKTokenizer(), Lemmatization(),
                                                      Encoder(is_split_into_words=True,
                                                              truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction_with_stemming',
                                            iterable=[NoiseReduction(), NLTKTokenizer(), Stemming(),
                                                      Encoder(is_split_into_words=True,
                                                              truncation_max_length=max_tokens)])
                      ])
# optional - for testing purposes, if you want to run fast test on very small datasets
# truncate(pipelines, fraction=0.05)

dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': pipelines})

print('Dataset Hyperparameters Grid Size: ', len(dataset_hparams_grid))

model_hparams_grid = ParameterGrid({'epochs': [10],
                                    'batch_size': [16],
                                    'learning_rate': [5e-05],
                                    'lr_scheduler': ['linear'],
                                    'optimizer': ['adamw_torch'],
                                    'weight_decay': [1e-3],
                                    'early_stopping_patience': [3],
                                    'early_stopping_threshold': [0.01]})


print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(dataset_hparams_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  1
Total Hyperparameter Combinations:  4


### ReCOVery Dataset

In [13]:
dataset_name = 'ReCOVery'
dataset_path = '../../sources/local_datasets/data/ReCOVery/recovery.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### Run 1 of 4

2025/05/16 01:02:48 INFO mlflow.tracking.fluent: Experiment with name 'prefinal-distilli_bert-v2' does not exist. Creating a new experiment.
2025-05-16 01:02:48,992 - Experiment - INFO - MLflow experiment initialized with ID: 701195748125061381
2025-05-16 01:02:48,993 - Experiment - INFO - Model signature: model(mn=distillibert,ti=distilbert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01)
2025-05-16 01:02:48,994 - Experiment - INFO - Dataset signature: dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),distilli_bert__encoder(t=distilbert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 01:02:48,994 - Experiment - INFO - Run signature: model(mn=distillibert,ti=distilbert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01);dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),distilli_bert__encoder(t=distilbert-base-uncased,t=true,p=max_l

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

2025-05-16 01:04:29,465 - Experiment - INFO - Test metrics: {'test_loss': 0.464690625667572, 'test_accuracy': 0.8, 'test_precision': 0.7857142857142857, 'test_recall': 1.0, 'test_f1_score': 0.88, 'test_roc_auc': 0.8636363636363636, 'test_false_positives_rate': 0.75, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.0888, 'test_samples_per_second': 168.957, 'test_steps_per_second': 11.264, 'test_epoch': 10.0}
2025-05-16 01:04:29,482 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:04:29,484 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:04:29,523 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:04:29,524 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-16 01:04:29,525 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-50
2025-05-16 01:04:29,780 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.3494666516780853, 'eval_accuracy': 0.866666666666

2025-05-16 01:04:29,880 - Experiment - INFO - Test metrics: {'test_loss': 0.464690625667572, 'test_accuracy': 0.8, 'test_precision': 0.7857142857142857, 'test_recall': 1.0, 'test_f1_score': 0.88, 'test_roc_auc': 0.8636363636363636, 'test_false_positives_rate': 0.75, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.0924, 'test_samples_per_second': 162.36, 'test_steps_per_second': 10.824, 'test_epoch': 10.0}
2025-05-16 01:04:29,908 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:04:29,911 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:04:29,961 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:04:29,962 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-16 01:04:29,962 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-50
2025-05-16 01:04:30,401 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.3494666516780853, 'eval_accuracy': 0.8

2025-05-16 01:04:30,500 - Experiment - INFO - Test metrics: {'test_loss': 0.464690625667572, 'test_accuracy': 0.8, 'test_precision': 0.7857142857142857, 'test_recall': 1.0, 'test_f1_score': 0.88, 'test_roc_auc': 0.8636363636363636, 'test_false_positives_rate': 0.75, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.0885, 'test_samples_per_second': 169.449, 'test_steps_per_second': 11.297, 'test_epoch': 10.0}
2025-05-16 01:04:30,524 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:04:30,528 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:04:30,576 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:04:30,577 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-16 01:04:30,579 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-50
2025-05-16 01:04:30,902 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.3494666516780853, 'eval_accuracy': 0.8666666666666

2025-05-16 01:04:31,005 - Experiment - INFO - Test metrics: {'test_loss': 0.464690625667572, 'test_accuracy': 0.8, 'test_precision': 0.7857142857142857, 'test_recall': 1.0, 'test_f1_score': 0.88, 'test_roc_auc': 0.8636363636363636, 'test_false_positives_rate': 0.75, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.0965, 'test_samples_per_second': 155.386, 'test_steps_per_second': 10.359, 'test_epoch': 10.0}
2025-05-16 01:04:31,027 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:04:31,029 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:04:31,080 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:04:31,081 - Experiment - INFO - Evaluating model using loss metric...
2025-05-16 01:04:31,081 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-50
2025-05-16 01:04:31,388 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.3494666516780853, 'eval_accuracy': 0.8666666666666667

2025-05-16 01:04:31,500 - Experiment - INFO - Test metrics: {'test_loss': 0.464690625667572, 'test_accuracy': 0.8, 'test_precision': 0.7857142857142857, 'test_recall': 1.0, 'test_f1_score': 0.88, 'test_roc_auc': 0.8636363636363636, 'test_false_positives_rate': 0.75, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.1035, 'test_samples_per_second': 144.897, 'test_steps_per_second': 9.66, 'test_epoch': 10.0}
2025-05-16 01:04:31,538 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:04:31,542 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:04:31,645 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:04:31,647 - Experiment - INFO - Finished model evaluations stage.


### EUvsDisinfo Dataset

In [ ]:
dataset_name = 'EUvsDisinfo'
dataset_path = '../../sources/local_datasets/data/EUvsDisinfo/euvsdisinfo.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### ISOT Dataset

In [ ]:
dataset_name = 'ISOT'
dataset_path = '../../sources/local_datasets/data/ISOT/isot.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### CZ Dataset

In [6]:
cz_pipelines = []

for max_tokens in max_tokens_params:
    cz_pipelines.extend([PreprocessingPipeline(name='minimal',
                                               iterable=[Encoder(truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction',
                                               iterable=[NoiseReduction(),
                                                         Encoder(truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                               iterable=[NoiseReduction(), NLTKTokenizer(language='czech'),
                                                         Lemmatization(language='czech'),
                                                         Encoder(is_split_into_words=True,
                                                                 truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction_with_stemming',
                                               iterable=[NoiseReduction(), NLTKTokenizer(language='czech'),
                                                         Stemming(language='czech'),
                                                         Encoder(is_split_into_words=True,
                                                                 truncation_max_length=max_tokens)])
                         ])
# optional - for testing purposes, if you want to run fast test on very small datasets
# truncate(cz_pipelines, fraction=0.1)

cz_dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': cz_pipelines})
print('Dataset Hyperparameters Grid Size: ', len(cz_dataset_hparams_grid))
print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(cz_dataset_hparams_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  1
Total Hyperparameter Combinations:  4


In [7]:
dataset_name = 'cz_dataset'
dataset_path = '../../sources/local_datasets/data/cz_dataset/cz_dataset.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          cz_dataset_hparams_grid,
                          model_hparams_grid)

### Run 1 of 4

2025-05-16 01:25:51,699 - Experiment - INFO - MLflow experiment initialized with ID: 701195748125061381
2025-05-16 01:25:51,700 - Experiment - INFO - Model signature: model(mn=distillibert,ti=distilbert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01)
2025-05-16 01:25:51,701 - Experiment - INFO - Dataset signature: dataset(dn=cz_dataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),distilli_bert__encoder(t=distilbert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 01:25:51,702 - Experiment - INFO - Run signature: model(mn=distillibert,ti=distilbert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01);dataset(dn=cz_dataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),distilli_bert__encoder(t=distilbert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 01:25:52,877 - Experiment - INFO - Run ID: 73ea8a711d184207afdf21a2f4ac9ba0
2025-05-16 01:25:52,883 

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

2025-05-16 01:27:27,025 - Experiment - INFO - Test metrics: {'test_loss': 0.4773598313331604, 'test_accuracy': 0.8, 'test_precision': 0.8888888888888888, 'test_recall': 0.8, 'test_f1_score': 0.8421052631578947, 'test_roc_auc': 0.8400000000000001, 'test_false_positives_rate': 0.2, 'test_false_negatives_rate': 0.2, 'test_runtime': 0.0679, 'test_samples_per_second': 220.762, 'test_steps_per_second': 14.717, 'test_epoch': 9.0}
2025-05-16 01:27:27,041 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:27:27,043 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:27:27,086 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:27:27,087 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-16 01:27:27,088 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-45
2025-05-16 01:27:27,301 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.4291217625141144, 'eval_accuracy': 0

2025-05-16 01:27:27,374 - Experiment - INFO - Test metrics: {'test_loss': 0.4773598313331604, 'test_accuracy': 0.8, 'test_precision': 0.8888888888888888, 'test_recall': 0.8, 'test_f1_score': 0.8421052631578947, 'test_roc_auc': 0.8400000000000001, 'test_false_positives_rate': 0.2, 'test_false_negatives_rate': 0.2, 'test_runtime': 0.0682, 'test_samples_per_second': 219.898, 'test_steps_per_second': 14.66, 'test_epoch': 9.0}
2025-05-16 01:27:27,392 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:27:27,394 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:27:27,435 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:27:27,436 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-16 01:27:27,437 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-50
2025-05-16 01:27:27,661 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.42455819249153137, 'eval_

2025-05-16 01:27:27,737 - Experiment - INFO - Test metrics: {'test_loss': 0.5134567618370056, 'test_accuracy': 0.7333333333333333, 'test_precision': 0.875, 'test_recall': 0.7, 'test_f1_score': 0.7777777777777778, 'test_roc_auc': 0.8400000000000001, 'test_false_positives_rate': 0.2, 'test_false_negatives_rate': 0.3, 'test_runtime': 0.0707, 'test_samples_per_second': 212.17, 'test_steps_per_second': 14.145, 'test_epoch': 10.0}
2025-05-16 01:27:27,760 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:27:27,762 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:27:27,802 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:27:27,803 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-16 01:27:27,803 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-25
2025-05-16 01:27:28,058 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.5456773042678833, 'eval_accuracy': 

2025-05-16 01:27:28,146 - Experiment - INFO - Test metrics: {'test_loss': 0.6494572758674622, 'test_accuracy': 0.4666666666666667, 'test_precision': 0.75, 'test_recall': 0.3, 'test_f1_score': 0.42857142857142855, 'test_roc_auc': 0.8, 'test_false_positives_rate': 0.2, 'test_false_negatives_rate': 0.7, 'test_runtime': 0.0822, 'test_samples_per_second': 182.505, 'test_steps_per_second': 12.167, 'test_epoch': 5.0}
2025-05-16 01:27:28,184 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:27:28,186 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:27:28,229 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:27:28,230 - Experiment - INFO - Evaluating model using loss metric...
2025-05-16 01:27:28,230 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-50
2025-05-16 01:27:28,443 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.42455819249153137, 'eval_accuracy': 0.785714285714285

2025-05-16 01:27:28,528 - Experiment - INFO - Test metrics: {'test_loss': 0.5134567618370056, 'test_accuracy': 0.7333333333333333, 'test_precision': 0.875, 'test_recall': 0.7, 'test_f1_score': 0.7777777777777778, 'test_roc_auc': 0.8400000000000001, 'test_false_positives_rate': 0.2, 'test_false_negatives_rate': 0.3, 'test_runtime': 0.0789, 'test_samples_per_second': 190.12, 'test_steps_per_second': 12.675, 'test_epoch': 10.0}
2025-05-16 01:27:28,544 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:27:28,545 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:27:28,584 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:27:28,585 - Experiment - INFO - Finished model evaluations stage.
